In [ ]:
%pip install "datasets==2.19.1"

In [7]:
import json
from datasets import load_dataset
from transformers import AutoTokenizer

# Load BioBERT's tokenizer directly from Hugging Face
print("Loading BioBERT Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained("dmis-lab/biobert-v1.1")

def build_datasets():
    print("Downloading BC5CDR dataset from Hugging Face...")
    dataset = load_dataset("tner/bc5cdr", split="test", trust_remote_code=True)
    
    tag_names = {
        0: "O",
        1: "B-Chemical",
        2: "B-Disease",
        3: "I-Disease",
        4: "I-Chemical"
    }
    
    test_easy = []
    test_hard = []
    
    print("Processing tokens, calculating lengths, and routing to Easy vs. Hard buckets...")
    
    for idx, row in enumerate(dataset):
        tokens = row['tokens']
        tags = row['tags']
        
        text = ""
        entities = []
        current_entity = None
        
        # Helper function to pack the metadata before saving the entity
        def finalize_entity(ent):
            clean_text = ent["text"].strip()
            # 1. Measure the character length (The Weight)
            ent["char_length"] = len(clean_text)
            
            # 2. Measure the token fragmentation (The Pieces)
            sub_tokens = tokenizer.tokenize(clean_text)
            ent["token_count"] = len(sub_tokens)
            
            return ent

        # Reconstruct the document string and calculate character offsets
        for token, tag_id in zip(tokens, tags):
            tag_name = tag_names[tag_id]
            
            start_offset = len(text)
            text += token
            end_offset = len(text)
            
            if tag_name.startswith("B-"):
                if current_entity:
                    entities.append(finalize_entity(current_entity))
                
                label = tag_name.split("-")[1] 
                current_entity = {
                    "label": label,
                    "text": token,
                    "start": start_offset,
                    "end": end_offset
                }
            elif tag_name.startswith("I-") and current_entity:
                current_entity["text"] += " " + token
                current_entity["end"] = end_offset
            else:
                if current_entity:
                    entities.append(finalize_entity(current_entity))
                    current_entity = None
            
            text += " " 
            
        if current_entity:
            entities.append(finalize_entity(current_entity))
            
        text = text.rstrip() 
        
        # Skip empty documents
        if len(entities) == 0:
            continue 
            
        # The Scientific Split
        # If ANY entity in the document shatters into 3+ tokens, the doc goes to Hard
        is_hard_doc = any(ent["token_count"] >= 3 for ent in entities)
        
        doc_record = {
            "doc_id": str(idx),
            "text": text,
            "true_entities": entities
        }
        
        if is_hard_doc:
            test_hard.append(doc_record)
        else:
            test_easy.append(doc_record)
            
    # Export to JSON
    print(f"Exporting {len(test_easy)} Everyday docs to test_easy.json")
    with open("test_easy.json", "w", encoding="utf-8") as f:
        json.dump(test_easy, f, indent=2)
        
    print(f"Exporting {len(test_hard)} Hard Latinate docs to test_hard.json")
    with open("test_hard.json", "w", encoding="utf-8") as f:
        json.dump(test_hard, f, indent=2)

    print("Data Architecture complete! Length and Token metadata are packed and ready.")

if __name__ == "__main__":
    build_datasets()

Loading BioBERT Tokenizer...


c:\Users\stell\anaconda3\envs\ml\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\stell\.cache\huggingface\hub\models--dmis-lab--biobert-v1.1. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Processing tokens, calculating lengths, and routing to Easy vs. Hard buckets...
Exporting 524 Everyday docs to test_easy.json
Exporting 3616 Hard Latinate docs to test_hard.json
Data Architecture complete! Length and Token metadata are packed and ready.
